# 🎯 K-Nearest Neighbors (KNN) - Implementación desde CERO

## Objetivos
- Entender el algoritmo KNN
- Implementar KNN desde cero
- Clasificación y regresión con KNN
- Elegir el valor óptimo de K

## Tabla de Contenidos
- [1 - Paquetes](#1)
- [2 - Conceptos Principales](#2)
- [3 - Ejercicios Prácticos](#3)
- [4 - Resumen](#4)

In [ ]:
# ==========================================
# CONFIGURACIÓN DEL ENTORNO
# ==========================================
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import sys
from pathlib import Path

# Agregar el directorio raíz al path de manera robusta
project_root = Path.cwd().parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Verificar que las utilidades se pueden importar
try:
    from utils.plot_utils import plot_decision_boundary
    print("✅ Entorno configurado correctamente")
    print(f"📁 Raíz del proyecto: {project_root}")
except ImportError as e:
    print("❌ Error al importar utilidades")
    print("\n💡 Soluciones:")
    print("   1. Ejecuta 'pip install -e .' desde la raíz del proyecto")
    print("   2. O inicia Jupyter desde la raíz: cd ML-FROM-ZERO-PYTHON && jupyter notebook")
    print(f"\n🔍 Error detallado: {e}")
    raise

## 1. Teoría de KNN

KNN es un algoritmo NO paramétrico (no aprende parámetros).

### Algoritmo:
1. Calcular distancia entre el punto nuevo y todos los puntos de entrenamiento
2. Encontrar los K vecinos más cercanos
3. **Clasificación**: Votar por la clase más común
4. **Regresión**: Promediar los valores de los K vecinos

### Distancia Euclidiana:
$$d(x_1, x_2) = \sqrt{\sum_{i=1}^{n} (x_{1i} - x_{2i})^2}$$

## 2. Implementación desde CERO

In [ ]:
class KNNClassifier:
    """
    K-Nearest Neighbors Classifier desde cero.
    """
    
    def __init__(self, k=3):
        self.k = k
        self.X_train = None
        self.y_train = None
    
    def fit(self, X, y):
        """Guardar datos de entrenamiento (KNN no 'entrena')"""
        self.X_train = np.array(X)
        self.y_train = np.array(y)
        return self
    
    def _euclidean_distance(self, x1, x2):
        """Calcula distancia euclidiana"""
        return np.sqrt(np.sum((x1 - x2) ** 2))
    
    def predict(self, X):
        """Predice clases para X"""
        X = np.array(X)
        if len(X.shape) == 1:
            X = X.reshape(1, -1)
        
        predictions = [self._predict_single(x) for x in X]
        return np.array(predictions)
    
    def _predict_single(self, x):
        """Predice clase para un único punto"""
        # Calcular distancias a todos los puntos de entrenamiento
        distances = [self._euclidean_distance(x, x_train) 
                    for x_train in self.X_train]
        
        # Obtener índices de los K vecinos más cercanos
        k_indices = np.argsort(distances)[:self.k]
        
        # Obtener las etiquetas de los K vecinos
        k_nearest_labels = self.y_train[k_indices]
        
        # Votar por la clase más común
        most_common = Counter(k_nearest_labels).most_common(1)
        return most_common[0][0]
    
    def score(self, X, y):
        """Calcula accuracy"""
        y_pred = self.predict(X)
        return np.mean(y_pred == y)

## 3. Ejemplo de Clasificación

In [ ]:
from sklearn.datasets import make_classification

# Generar datos
X, y = make_classification(n_samples=100, n_features=2, n_redundant=0,
                          n_informative=2, n_clusters_per_class=1, 
                          random_state=42)

# Train/Test split
def train_test_split(X, y, test_size=0.2, random_state=42):
    np.random.seed(random_state)
    n = len(X)
    indices = np.random.permutation(n)
    test_size_n = int(n * test_size)
    test_idx = indices[:test_size_n]
    train_idx = indices[test_size_n:]
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

X_train, X_test, y_train, y_test = train_test_split(X, y)

print(f"Train: {len(X_train)}, Test: {len(X_test)}")

In [ ]:
# Entrenar KNN
knn = KNNClassifier(k=5)
knn.fit(X_train, y_train)

# Evaluar
train_acc = knn.score(X_train, y_train)
test_acc = knn.score(X_test, y_test)

print(f"Training Accuracy: {train_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

In [ ]:
# Visualizar frontera de decisión
plot_decision_boundary(X_train, y_train, knn, 
                      title=f"KNN (k={knn.k}) - Frontera de Decisión")
plt.show()

## 4. Elegir el Valor Óptimo de K

In [ ]:
# Probar diferentes valores de K
k_values = range(1, 21)
train_scores = []
test_scores = []

for k in k_values:
    knn_temp = KNNClassifier(k=k)
    knn_temp.fit(X_train, y_train)
    train_scores.append(knn_temp.score(X_train, y_train))
    test_scores.append(knn_temp.score(X_test, y_test))

# Visualizar
plt.figure(figsize=(10, 6))
plt.plot(k_values, train_scores, 'o-', label='Train', linewidth=2)
plt.plot(k_values, test_scores, 'o-', label='Test', linewidth=2)
plt.xlabel('Valor de K')
plt.ylabel('Accuracy')
plt.title('Accuracy vs K')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

best_k = k_values[np.argmax(test_scores)]
print(f"\nMejor K: {best_k}")
print(f"Test Accuracy con K={best_k}: {max(test_scores):.4f}")

## 🎯 Ejercicio: Implementa KNN para Regresión

KNN también funciona para regresión, promediando los valores de los K vecinos.

In [ ]:
# TU CÓDIGO AQUÍ
class KNNRegressor:
    def __init__(self, k=3):
        self.k = k
    
    def fit(self, X, y):
        # Tu código aquí
        pass
    
    def predict(self, X):
        # En lugar de votar, promediar los valores de los K vecinos
        pass

## 🎓 Resumen

- ✅ KNN es simple pero efectivo
- ✅ No paramétrico (no entrena)
- ✅ Puede ser lento en predicción con datasets grandes
- ✅ Sensible a la escala de features (normalizar!)
- ✅ El valor de K es importante

### Próximo: K-Means Clustering